In [ ]:
import sys
import os
sys.path.append(os.path.abspath('..'))  # Add parent directory to path

import numpy as np
from diffusion_geometry.visualisation import *
from plotly.subplots import make_subplots

In [ ]:
n_points = 15
x = np.linspace(-1, 1, n_points)
y = np.linspace(-1, 1, n_points)
x, y = np.meshgrid(x, y)
data = np.vstack([x.ravel(), y.ravel()]).T
n = data.shape[0]

nabla_x = np.array([[1,0]]*n)
nabla_y = np.array([[0,1]]*n)

quiver_scale = 0.1
quiver_line_width = 1
quiver_arrow_scale = 0.5
fig_nabla_x = plot_quiver_2d(data, nabla_x, scale=quiver_scale, line_width=quiver_line_width, arrow_scale=quiver_arrow_scale)
fig_nabla_y = plot_quiver_2d(data, nabla_y, scale=quiver_scale, line_width=quiver_line_width, arrow_scale=quiver_arrow_scale)
fig_nabla_x.show()
fig_nabla_y.show()

def f_func(x,y):
    return np.sin(1*x)*np.cos(3*y) + 0.5*x**3 + 0.5

def h_func(x,y):
    return np.sin(x + y)*np.cos(3*y)

x_grid = np.linspace(-1, 1, 100)
y_grid = np.linspace(-1, 1, 100)
X_grid, Y_grid = np.meshgrid(x_grid, y_grid)

fig_f = plot_heatmap_2d(X_grid, Y_grid, f_func(X_grid, Y_grid))
fig_h = plot_heatmap_2d(X_grid, Y_grid, h_func(X_grid, Y_grid))

fig_f.show()
fig_h.show()

f_nabla_x = f_func(x, y).ravel().reshape(-1, 1) * nabla_x
h_nabla_y = h_func(x, y).ravel().reshape(-1, 1) * nabla_y
fig_f_nabla_x = plot_quiver_2d(data, f_nabla_x, scale=quiver_scale, line_width=quiver_line_width, arrow_scale=quiver_arrow_scale)
fig_h_nabla_y = plot_quiver_2d(data, h_nabla_y, scale=quiver_scale, line_width=quiver_line_width, arrow_scale=quiver_arrow_scale)
fig_f_nabla_x.show()
fig_h_nabla_y.show()

fig_X = plot_quiver_2d(data, f_nabla_x + h_nabla_y, scale=quiver_scale, line_width=2*quiver_line_width, arrow_scale=quiver_arrow_scale)
fig_X.show()


In [ ]:
n_theta = 50
theta = np.linspace(0, 2*np.pi, n_theta, endpoint=False)
theta_dense = np.linspace(0, 2*np.pi, 500)

# Curve
rescale_x = 0.8
rescale_y = 0.9
offset_x = -0.3
x = rescale_x*(np.cos(theta) + 0.5*np.cos(2*theta)) + offset_x
y = rescale_y*(np.sin(theta))
circle = np.c_[x, y]
circle_dense = np.c_[rescale_x*(np.cos(theta_dense) + 0.5*np.cos(2*theta_dense)) + offset_x, rescale_y*(np.sin(theta_dense))]

# Tangent and unit tangent
dx = -np.sin(theta) - np.sin(2*theta)   # derivative of cos + 0.5*cos(2θ)
dy =  np.cos(theta)
t = np.c_[rescale_x*dx, rescale_y*dy]
t_norm = np.linalg.norm(t, axis=1, keepdims=True)
t_hat = t / t_norm  # (n,2)

# Ambient coordinate gradients
ex = np.array([1.0, 0.0])  # ∇x
ey = np.array([0.0, 1.0])  # ∇y

# Project onto tangent: (v·t̂) t̂
circle_nabla_x = (t_hat @ ex)[:, None] * t_hat   # (n,2)
circle_nabla_y = (t_hat @ ey)[:, None] * t_hat   # (n,2)

fig_circle_nabla_x = plot_quiver_2d(circle, circle_nabla_x, scale=quiver_scale, line_width=quiver_line_width, arrow_scale=quiver_arrow_scale)
fig_circle_nabla_y = plot_quiver_2d(circle, circle_nabla_y, scale=quiver_scale, line_width=quiver_line_width, arrow_scale=quiver_arrow_scale)
fig_circle_nabla_x.show()
fig_circle_nabla_y.show()

# --- Utility: overlay the curve outline ---
def add_curve_trace(fig, curve, color="black", width=1.5, opacity=1.0):
    fig.add_trace(
        go.Scatter(
            x=curve[:, 0],
            y=curve[:, 1],
            mode="lines",
            line=dict(color=color, width=width),
            opacity=opacity,
            showlegend=False,
        )
    )
    return fig

f_vec_circle = f_func(circle_dense[:,0], circle_dense[:,1]).ravel()
fig_circle_f = plot_scatter_2d(circle_dense, f_vec_circle)
fig_circle_f.show()

h_vec_circle = h_func(circle_dense[:,0], circle_dense[:,1]).ravel()
fig_circle_h = plot_scatter_2d(circle_dense, h_vec_circle)
fig_circle_h.show()

circle_f_nabla_x = f_func(circle[:,0], circle[:,1]).ravel().reshape(-1, 1) * circle_nabla_x
circle_h_nabla_y = h_func(circle[:,0], circle[:,1]).ravel().reshape(-1, 1) * circle_nabla_y
fig_circle_f_nabla_x = plot_quiver_2d(circle, circle_f_nabla_x, scale=quiver_scale, line_width=quiver_line_width, arrow_scale=quiver_arrow_scale)
fig_circle_h_nabla_y = plot_quiver_2d(circle, circle_h_nabla_y, scale=quiver_scale, line_width=quiver_line_width, arrow_scale=quiver_arrow_scale)
fig_circle_f_nabla_x.show()
fig_circle_h_nabla_y.show()
fig_circle_X = plot_quiver_2d(circle, circle_f_nabla_x + circle_h_nabla_y, scale=quiver_scale, line_width=2*quiver_line_width, arrow_scale=quiver_arrow_scale)
fig_circle_X.show()

In [ ]:
from plotly.subplots import make_subplots

# ---------- helpers ----------
def _axis_suffix_map(specs):
    """Map (row,col) → axis suffix ('' | '2' | '3' ...), respecting colspan and None."""
    m, idx = {}, 1
    for r, row in enumerate(specs, 1):
        c = 1
        while c <= len(row):
            cell = row[c - 1]
            if cell is None:
                c += 1
                continue
            span = int(cell.get("colspan", 1)) if isinstance(cell, dict) else 1
            typ = cell.get("type", "xy") if isinstance(cell, dict) else "xy"
            if typ == "xy":
                suf = "" if idx == 1 else str(idx)
                m[(r, c)] = suf
                idx += 1
            c += span
    return m

def _xref(suffix):  # convert suffix to full axis name
    return "x" if suffix == "" else f"x{suffix}"

def _add_all(dst, src_fig, row, col):
    for tr in src_fig.data:
        dst.add_trace(tr, row=row, col=col)

def _hide_axes(fig):
    """Hide ticks, labels, gridlines, and zero lines on all 2D subplots."""
    fig.update_xaxes(
        showgrid=False, showline=False, showticklabels=False,
        zeroline=False, title_text="", mirror=False
    )
    fig.update_yaxes(
        showgrid=False, showline=False, showticklabels=False,
        zeroline=False, title_text="", mirror=False
    )

# ---------- 1) compute common ranges ----------
xr = getattr(fig_nabla_x.layout.xaxis, "range", None) or [-1, 1]
yr = getattr(fig_nabla_x.layout.yaxis, "range", None) or [-1, 1]
xr2 = getattr(fig_nabla_y.layout.xaxis, "range", None)
yr2 = getattr(fig_nabla_y.layout.yaxis, "range", None)
if xr2 and yr2:
    xr = [min(xr[0], xr2[0]), max(xr[1], xr2[1])]
    yr = [min(yr[0], yr2[0]), max(yr[1], yr2[1])]
xmax = ymax = max(abs(x) for x in (*xr, *yr))

# ---------- 2) layout ----------
specs = [
    [{"type": "xy"}, {"type": "xy"}, {"type": "xy"}, {"type": "xy"}],
    [{"type": "xy"}, {"type": "xy"}, {"type": "xy"}, {"type": "xy"}],
    [{"type": "xy"}, {"type": "xy"}, {"type": "xy"}, {"type": "xy"}],
    [{"colspan": 2, "type": "xy"}, None, {"colspan": 2, "type": "xy"}, None],
    [None, None, None, None],
]
fig_all = make_subplots(
    rows=5, cols=4,
    specs=specs,
    row_heights=[0.18, 0.18, 0.18, 0.36, 0.0],
    vertical_spacing=0.04, horizontal_spacing=0.0,
)

# ---------- 3) add traces ----------
# left block
for (r, c), f in {
    (1, 1): fig_nabla_x, (1, 2): fig_nabla_y,
    (2, 1): fig_f, (2, 2): fig_h,
    (3, 1): fig_f_nabla_x, (3, 2): fig_h_nabla_y,
}.items():
    _add_all(fig_all, f, r, c)
_add_all(fig_all, fig_X, 4, 1)

# right block
for (r, c), f in {
    (1, 3): fig_circle_nabla_x, (1, 4): fig_circle_nabla_y,
    (2, 3): fig_circle_f, (2, 4): fig_circle_h,
    (3, 3): fig_circle_f_nabla_x, (3, 4): fig_circle_h_nabla_y,
}.items():
    _add_all(fig_all, f, r, c)
_add_all(fig_all, fig_circle_X, 4, 3)

# ---------- 4) link groups & ranges ----------
# Left group
fig_all.update_xaxes(range=[-xmax, xmax], row=1, col=1)
fig_all.update_yaxes(range=[-xmax, xmax], row=1, col=1)
for r, c in [(1, 2), (2, 1), (2, 2), (3, 1), (3, 2), (4, 1)]:
    fig_all.update_xaxes(matches="x", row=r, col=c)
    fig_all.update_yaxes(matches="y", row=r, col=c)

# Right group
fig_all.update_xaxes(range=[-xmax, xmax], row=1, col=3)
fig_all.update_yaxes(range=[-xmax, xmax], row=1, col=3)
for r, c in [(1, 4), (2, 3), (2, 4), (3, 3), (3, 4), (4, 3)]:
    fig_all.update_xaxes(matches="x3", row=r, col=c)
    fig_all.update_yaxes(matches="y3", row=r, col=c)

# ---------- 5) enforce 1:1 ratio ----------
suffix = _axis_suffix_map(specs)
for (r, c), s in suffix.items():
    fig_all.update_yaxes(scaleanchor=_xref(s), scaleratio=1, row=r, col=c)

# Keep axes square and domains consistent
fig_all.update_xaxes(constrain="domain")
fig_all.update_yaxes(constrain="domain")

# ---------- 6) cosmetics ----------
_hide_axes(fig_all)
fig_all.update_layout(
    width=800, height=1000,
    margin=dict(l=0, r=0, t=0, b=0),
    plot_bgcolor="white", paper_bgcolor="white",
    showlegend=False, title="",
)

# ---------- display / export ----------
fig_all.show()

import plotly.io as pio
pio.write_image(fig_all, "figs/intro_figs_clean.pdf", scale=1)


In [ ]:
def labels(row, col):
    if row == 1:
        if col in (1, 3):
            return r"$\nabla x$"
        elif col in (2, 4):
            return r"$\nabla y$"
    elif row == 2:
        if col in (1, 3):
            return r"$f$"
        elif col in (2, 4):
            return r"$h$"
    elif row == 3:
        if col in (1, 3):
            return r"$f \nabla x$"
        elif col in (2, 4):
            return r"$h \nabla y$"
    elif row == 4:
        return r"$X = f \nabla x + h \nabla y$"
    else:
        return ""

overpic_labels(fig_all, labels, 
                       stretch_x=0.98,
                       stretch_y=1,
                       offset_y=+10)

In [ ]:
n_theta = 70
theta = np.linspace(0, 2*np.pi, n_theta, endpoint=False)
theta_dense = np.linspace(0, 2*np.pi, 500)

# Curve
rescale_x = 0.8
rescale_y = 0.9
offset_x = -0.3
x = rescale_x*(np.cos(theta) + 0.5*np.cos(2*theta)) + offset_x
y = rescale_y*(np.sin(theta))
circle = np.c_[x, y]
circle_dense = np.c_[rescale_x*(np.cos(theta_dense) + 0.5*np.cos(2*theta_dense)) + offset_x, rescale_y*(np.sin(theta_dense))]

# Tangent and unit tangent
dx = -np.sin(theta) - np.sin(2*theta)   # derivative of cos + 0.5*cos(2θ)
dy =  np.cos(theta)
t = np.c_[rescale_x*dx, rescale_y*dy]
t_norm = np.linalg.norm(t, axis=1, keepdims=True)
t_hat = t / t_norm  # (n,2)

# Ambient coordinate gradients
ex = np.array([1.0, 0.0])  # ∇x
ey = np.array([0.0, 1.0])  # ∇y

# Project onto tangent: (v·t̂) t̂
circle_nabla_x = (t_hat @ ex)[:, None] * t_hat   # (n,2)
circle_nabla_y = (t_hat @ ey)[:, None] * t_hat   # (n,2)


def f_func(x,y):
    return np.sin(1*x)*np.cos(3*y) + 0.5*x**3 + 0.5

def grad_f_func(x,y):
    dfdx = np.cos(1*x)*np.cos(3*y) + 1.5*x**2
    dfdy = -3*np.sin(1*x)*np.sin(3*y)
    return np.c_[dfdx, dfdy]

f_sample_circle = f_func(circle[:,0], circle[:,1]).ravel()
fig_circle_sample_f = plot_scatter_2d(circle, f_sample_circle)
fig_circle_sample_f.show()

grad_f_vec = grad_f_func(data[:,0], data[:,1])   # (n,2)
fig_grad_f = plot_quiver_2d(data, grad_f_vec, scale=quiver_scale, line_width=quiver_line_width, arrow_scale=quiver_arrow_scale)
fig_grad_f.show()

# project onto tangent: (v·t̂) t̂
circle_gamma = np.stack([circle_nabla_x, circle_nabla_y], axis=1)   # (n,2,2)
circle_grad_f = np.einsum('ijk,ik->ij', circle_gamma, grad_f_func(circle[:,0], circle[:,1]))   # (n,2)
fig_circle_grad_f = plot_quiver_2d(circle, circle_grad_f, scale=quiver_scale, line_width=quiver_line_width, arrow_scale=quiver_arrow_scale)
fig_circle_grad_f.show()

# diffusion geometry figure
from diffusion_geometry import DiffusionGeometry
dg = DiffusionGeometry.from_point_cloud(circle, bandwidth_variability=0.3)
f_dg = dg.function(f_sample_circle)
circle_dg_grad_f = f_dg.grad().to_ambient()
fig_circle_dg_grad_f = plot_quiver_2d(circle, circle_dg_grad_f, scale=quiver_scale, line_width=quiver_line_width, arrow_scale=quiver_arrow_scale)
fig_circle_dg_grad_f.show()


In [ ]:

theta = np.linspace(-1.7, 0, 12, endpoint=False)

# Curve
x = - 0.9*np.cos(theta) + 0.6
y = - 0.8*np.sin(theta) - 0.45
curve = np.c_[x, y]

# Splat
splat = np.random.normal(size=[60,2]) * 0.2 - [0.5, 0.3]

# Join together
non_manifold = np.r_[circle, curve, splat]

plot_scatter_2d(non_manifold).show()


# sampled function figure
f_non_manifold = f_func(non_manifold[:,0], non_manifold[:,1]).ravel()
fig_non_manifold_f = plot_scatter_2d(non_manifold, f_non_manifold)
fig_non_manifold_f.show()

# diffusion geometry figure
dg = DiffusionGeometry.from_point_cloud(non_manifold, bandwidth_variability=0.3)
f_non_manifold_dg = dg.function(f_non_manifold)
circle_dg_grad_f = f_non_manifold_dg.grad().to_ambient()
fig_non_manifold_dg_grad_f = plot_quiver_2d(non_manifold, circle_dg_grad_f, scale=quiver_scale, line_width=quiver_line_width, arrow_scale=quiver_arrow_scale)
fig_non_manifold_dg_grad_f.show()

In [ ]:
from plotly.subplots import make_subplots

# ---------- helpers ----------
def _axis_suffix_map(specs):
    """Map (row,col) → axis suffix ('' | '2' | '3' ...), respecting colspan and None."""
    m, idx = {}, 1
    for r, row in enumerate(specs, 1):
        c = 1
        while c <= len(row):
            cell = row[c - 1]
            if cell is None:
                c += 1
                continue
            span = int(cell.get("colspan", 1)) if isinstance(cell, dict) else 1
            typ = cell.get("type", "xy") if isinstance(cell, dict) else "xy"
            if typ == "xy":
                suf = "" if idx == 1 else str(idx)
                m[(r, c)] = suf
                idx += 1
            c += span
    return m

def _xref(suffix):  # convert suffix to full axis name
    return "x" if suffix == "" else f"x{suffix}"

def _add_all(dst, src_fig, row, col):
    for tr in src_fig.data:
        dst.add_trace(tr, row=row, col=col)

def _hide_axes(fig):
    """Hide ticks, labels, gridlines, and zero lines on all 2D subplots."""
    fig.update_xaxes(
        showgrid=False, showline=False, showticklabels=False,
        zeroline=False, title_text="", mirror=False
    )
    fig.update_yaxes(
        showgrid=False, showline=False, showticklabels=False,
        zeroline=False, title_text="", mirror=False
    )

# ---------- 1) compute common ranges ----------
xr = getattr(fig_nabla_x.layout.xaxis, "range", None) or [-1, 1]
yr = getattr(fig_nabla_x.layout.yaxis, "range", None) or [-1, 1]
xr2 = getattr(fig_nabla_y.layout.xaxis, "range", None)
yr2 = getattr(fig_nabla_y.layout.yaxis, "range", None)
if xr2 and yr2:
    xr = [min(xr[0], xr2[0]), max(xr[1], xr2[1])]
    yr = [min(yr[0], yr2[0]), max(yr[1], yr2[1])]
xmax = ymax = max(abs(x) for x in (*xr, *yr))

# ---------- 2) layout ----------
specs = [
    [{"type": "xy"}, {"type": "xy"}, {"type": "xy"}, {"type": "xy"}],
    [{"type": "xy"}, {"type": "xy"}, {"type": "xy"}, {"type": "xy"}],
]
fig_all = make_subplots(
    rows=2, cols=4,
    specs=specs,
    row_heights=[0.18, 0.18],
    vertical_spacing=0.08, horizontal_spacing=0.0,
)

# ---------- 3) add traces ----------
# left block
for (r, c), f in {
    (1, 1): fig_f, (1, 2): fig_circle_f,
    (2, 1): fig_grad_f, (2, 2): fig_circle_grad_f,
}.items():
    _add_all(fig_all, f, r, c)

# right block
for (r, c), f in {
    (1, 3): fig_circle_sample_f, (1, 4): fig_non_manifold_f,
    (2, 3): fig_circle_dg_grad_f, (2, 4): fig_non_manifold_dg_grad_f,
}.items():
    _add_all(fig_all, f, r, c)

# ---------- 4) link groups & ranges ----------
# Left group
fig_all.update_xaxes(range=[-xmax, xmax], row=1, col=1)
fig_all.update_yaxes(range=[-xmax, xmax], row=1, col=1)
for r, c in [(1, 2), (2, 1), (2, 2)]:
    fig_all.update_xaxes(matches="x", row=r, col=c)
    fig_all.update_yaxes(matches="y", row=r, col=c)

# Right group
fig_all.update_xaxes(range=[-xmax, xmax], row=1, col=3)
fig_all.update_yaxes(range=[-xmax, xmax], row=1, col=3)
for r, c in [(1, 4), (2, 3), (2, 4)]:
    fig_all.update_xaxes(matches="x3", row=r, col=c)
    fig_all.update_yaxes(matches="y3", row=r, col=c)

# ---------- 5) enforce 1:1 ratio ----------
suffix = _axis_suffix_map(specs)
for (r, c), s in suffix.items():
    fig_all.update_yaxes(scaleanchor=_xref(s), scaleratio=1, row=r, col=c)

# Keep axes square and domains consistent
fig_all.update_xaxes(constrain="domain")
fig_all.update_yaxes(constrain="domain")

# ---------- 6) cosmetics ----------
_hide_axes(fig_all)
fig_all.update_layout(
    width=800, height=400,
    margin=dict(l=0, r=0, t=0, b=0),
    plot_bgcolor="white", paper_bgcolor="white",
    showlegend=False, title="",
)

# ---------- display / export ----------
fig_all.show()

import plotly.io as pio
pio.write_image(fig_all, "figs/new_output.pdf", scale=1)


In [ ]:
def labels(row, col):
    if not row == 1:
        return ""
    if col == 1:
        return "$\R^2$"
    elif col == 2:
        return "$\M$"
    elif col == 3:
        return "Sample from $\M$"
    elif col == 4:
        return "Non-manifold"

overpic_labels(fig_all, labels, 
                       stretch_x=0.98,
                       stretch_y=1,
                       offset_y=-14,
                       offset_x=-1)